# Demo 1.1: GoalScope on open-ended code

**Insight.** The goal heads were found, and GoalScope evaluated, on prompts that number their sub-tasks (Demo 1). With
no steps listed at all, the goal heads still carry the step the LM is about to write, often its purpose rather than the
syntax of the next line.

**Setup.** The two prompts of the paper's figures, *"Implement the stable marriage algorithm in python"* and
*"Implement the merge-sort algorithm in python"*, with the assistant turn prefilled up to the function name. The LM
writes the rest greedily. GoalScope, as in Demo 1 (the 122 goal heads, the neutral probe *"What are you doing at this
line of the code?"*), reads out every token of the function.

**Paper.** App. `app:unspecified` (`fig:merge-sort`, `fig:stable-marriage`).

Needs one 80 GB GPU (Qwen2.5-Coder-32B-Instruct in bf16). Reading out every token takes a while (one 64-token answer
per token). The executed outputs are saved in this file.

In [1]:
# repo root = first parent dir that contains src/
import os, sys
_d = os.getcwd()
while _d != '/' and not os.path.isdir(os.path.join(_d, 'src')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)
print('repo root:', _d)

# no progress bars: keeps tqdm / HF widget outputs out of the saved notebook
os.environ['TQDM_DISABLE'] = '1'
import functools, tqdm.std, tqdm.notebook, transformers, huggingface_hub
for _c in (tqdm.std.tqdm, tqdm.notebook.tqdm):   # tqdm.notebook ignores TQDM_DISABLE, so force disable=True
    _c.__init__ = functools.partialmethod(_c.__init__, disable=True)
transformers.utils.logging.disable_progress_bar()
transformers.utils.logging.set_verbosity_error()      # no generation-config warnings
huggingface_hub.utils.disable_progress_bars()

# colors as in Demo 4: the LM's own code green, GoalScope-steered output purple, pivot token yellow
from typing import Literal
ide_theme: Literal['dark', 'light'] = 'dark'
if ide_theme == 'dark':   # light shades; the task-span fill reuses the text color
    DST_C, PAT_C = 'sea_green3', 'plum2'
    DST_BG       = DST_C
    PIVOT_BG     = 'bright_yellow'
else:                     # the paper's colors (Figure 5)
    DST_C, PAT_C = '#38761d', '#5b277d'
    DST_BG       = '#d9ead3'
    PIVOT_BG     = '#ffe6b2'

repo root: /disk/u/arnab/Codes/Projects/goals


In [2]:
CUDA         = '7'
MODEL        = 'Qwen/Qwen2.5-Coder-32B-Instruct'
GOAL_HEADS   = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal__lam20_thr0.9_band25-56.json'   # the 122 goal heads
ALPHA        = 0.1          # GoalScope strength, as in Demo 1
STEER_LAYERS = (35, 46)     # the probe is steered at layers 35..45
READOUT_LEN  = 64           # tokens of each GoalScope answer
SM_GEN       = 512          # stable marriage: tokens the LM may write
MS_GEN       = 256          # merge sort: tokens the LM may write
SEED         = 0

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

## Load the model
Qwen2.5-Coder-32B-Instruct and its 122 goal heads.

In [3]:
import time, warnings, html as _html
import torch
from rich.console import Console
from rich.table import Table
from rich.markup import escape
from IPython.display import HTML, display

from src.models import ModelandTokenizer
from src.utils import experiment_utils
from src.goals.head_sets import load_json_heads
from src.goals.goalscope import apply_goalscope
from src.goals.goalscope_viz import goalscope_qualitative_html
from src.tokens import prepare_input, find_token_range
import scripts.run_goalscope_fig32_stable_marriage as FIG   # the figure's prompt and callouts

console = Console(width=130, highlight=False)
torch.set_grad_enabled(False)
experiment_utils.set_seed(SEED)
os.makedirs('notebooks/demos/outputs', exist_ok=True)

warnings.filterwarnings('ignore', message='Consider using IPython.display.IFrame')
def iframe(doc, h=620):
    display(HTML(f'<iframe srcdoc="{_html.escape(doc, quote=True)}" style="width:100%;height:{h}px;'
                 f'border:1px solid #e5e7eb;border-radius:10px;" sandbox="allow-scripts"></iframe>'))

def greedy(prompt, n):
    inp = prepare_input(prompts=prompt, tokenizer=mt)
    g = mt._model.generate(input_ids=inp['input_ids'], attention_mask=inp['attention_mask'],
                           max_new_tokens=n, do_sample=False)
    gen = mt.tokenizer.decode(g[0, inp['input_ids'].shape[1]:], skip_special_tokens=True)
    return gen.split('</code>', 1)[0] + '</code>' if '</code>' in gen else gen   # keep the closed code block

def tokens_of(text):
    """token strings of `text` and their char starts; the code segment starts where apply_goalscope scopes it"""
    enc = prepare_input(prompts=text, tokenizer=mt, return_offsets_mapping=True)
    ids = enc['input_ids'][0].tolist()
    toks = [mt.tokenizer.decode([i]) for i in ids]
    starts, s = [], 0
    for t in toks:
        starts.append(s); s += len(t)
    code_start = find_token_range(string=text, substring='<code>', tokenizer=mt.tokenizer, occurrence=1,
                                  offset_mapping=enc['offset_mapping'][0])[0]
    return toks, starts, code_start

def line_start_tokens(toks, starts, text, lo, hi):
    """for each code line in tokens [lo, hi): the indentation token right before the line's first character"""
    out = []
    for j in range(lo, hi):
        t = toks[j]
        if t.strip() == '' and not t.endswith('\n') and j + 1 < hi and toks[j - 1].endswith('\n'):
            out.append(j)
    return out

def show_code(title, prefill, gen):
    console.rule(f'[bold {DST_C}]{title}[/]')
    console.print(f'[grey50]{escape(prefill)}[/][{DST_C}]{escape(gen.rstrip())}[/]')

T0 = time.time()
mt = ModelandTokenizer(model_key=MODEL, dtype=torch.bfloat16, device_map={'': 0}, attn_implementation='eager')
heads = load_json_heads(GOAL_HEADS, mt.n_layer, mt.n_heads)
GS = dict(strategy='additive_norm', alpha=ALPHA, steer_layer_range=STEER_LAYERS, n_steer_tokens='all',
          max_new_tokens=READOUT_LEN)
print(f'goal heads: {len(heads)}  (load {time.time() - T0:.0f}s)')

Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


goal heads: 122  (load 26s)


## 1. Stable marriage
The paper's prompt, prefilled up to the function signature. The LM writes the rest.

In [4]:
t0 = time.time()
prompt_a = (f"<|im_start|>system\n{FIG.SYSTEM}<|im_end|>\n<|im_start|>user\n{FIG.USER}<|im_end|>\n"
            f"<|im_start|>assistant\n{FIG.PREFILL}")
gen_a = greedy(prompt_a, SM_GEN)
show_code('the LM writes the function', FIG.PREFILL.removeprefix('<code>\n'), gen_a)
print(f'({time.time() - t0:.0f}s)')

─────────────────────────────────────────────────── the LM writes the function ───────────────────────────────────────────────────

def stable_marriage(men_preferences, women_preferences):
    n = len(men_preferences)
    men_free = list(range(n))
    women_free = list(range(n))
    men_engaged_to = [None] * n
    women_engaged_to = [None] * n
    men_proposals = [0] * n

    while men_free:
        man = men_free.pop(0)
        woman = men_preferences[man][men_proposals[man]]
        men_proposals[man] += 1

        if women_engaged_to[woman] is None:
            women_engaged_to[woman] = man
            men_engaged_to[man] = woman
        else:
            current_guy = women_engaged_to[woman]
            if women_preferences[woman].index(man) < women_preferences[woman].index(current_guy):
                women_engaged_to[woman] = man
                men_engaged_to[man] = woman
                men_free.append(current_guy)
            else:
                men_free.append(man)

    return list(zip(men_engaged_to, women_engaged_to))

# Example usage:
men_prefs = [
    [1, 0, 2],
    [0, 2, 1],
    [1, 2, 0]
]

women_prefs = [
    [2, 1, 0],
    [0, 2, 1],
    [1, 0, 2]
]

print(stable_marriage(men_prefs, women_prefs))
</code>

(26s)


### GoalScope at every token of the function
We read out every token from `<code>` to the end of the function's `return` line. The example usage the LM appends
after the function is skipped, as in the figure.

In [5]:
t0 = time.time()
full_a = prompt_a + gen_a
toks_a, starts_a, cs_a = tokens_of(full_a)
end_char = full_a.index(FIG.FIG_CODE.rstrip().splitlines()[-1]) + len(FIG.FIG_CODE.rstrip().splitlines()[-1])
fe_a = max(j for j, s in enumerate(starts_a) if s < end_char)          # token holding the end of `return ...`
pos_a = list(range(cs_a, fe_a + 1))
res_a = apply_goalscope(full_a, mt=mt, selected_heads=heads, include_noncode=True, token_positions=pos_a, **GS)
ro_a = {r['pos']: (r['patched_gen'] or '').strip() for r in res_a if r['scoped']}
print(f'{len(ro_a)} read-outs ({time.time() - t0:.0f}s)')

237 read-outs (325s)


### The figure's four callouts
Each callout of the paper's figure sits on the indentation right before a line, before the LM has written anything of
that line. Our readout at each, next to the figure's.

In [6]:
marks = []                                   # (callout, line, pos, figure read-out)
for num, line, occ, ref in FIG.FIG_READOUTS:
    idx = -1
    for _ in range(occ + 1):
        idx = full_a.find(line, idx + 1)
    j = max(i for i, s in enumerate(starts_a) if s < idx)
    j -= toks_a[j].strip() != '' and full_a[starts_a[j]:idx].strip() == ''
    marks.append((num, line, j, ref))

t = Table(title='stable marriage: GoalScope at the figure callouts', show_lines=True)
t.add_column('#'); t.add_column('next line', max_width=30)
t.add_column('GoalScope readout', max_width=45); t.add_column('readout in the figure', max_width=45)
for num, line, p, ref in sorted(marks):
    t.add_row(str(num), escape(line), f'[{PAT_C}]{escape(ro_a[p])}[/]', escape(ref))
console.print(t)

                                        stable marriage: GoalScope at the figure callouts                                         
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ next line                      ┃ GoalScope readout                           ┃ readout in the figure                       ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ men_engaged_to[man] = woman    │ I'm assigning the current index to the      │ I'm assigning the current index to the      │
│   │                                │ variable representing the person being      │ variable representing the person being      │
│   │                                │ matched.                                    │ matched.                                    │
├───┼────────────────────────────────┼─────────────────────────────────────────────┼─────────────────────────────────────────────┤
│ 2 │ man = men_free.pop(0)          │ I'm trying to match an unmatched man with a │ I'm attempting to match an unmatched man    │
│   │                                │ woman he hasn't proposed to yet.            │ with a woman he hasn't proposed to yet.     │
├───┼────────────────────────────────┼─────────────────────────────────────────────┼─────────────────────────────────────────────┤
│ 3 │ if                             │ I'm evaluating which option is preferred    │ I'm evaluating which option is preferred    │
│   │ women_preferences[woman].inde… │ based on the current preferences list.      │ based on the current preferences list.      │
├───┼────────────────────────────────┼─────────────────────────────────────────────┼─────────────────────────────────────────────┤
│ 4 │ men_free.append(current_guy)   │ I'm updating the reference to point to a    │ I'm updating the reference to point to a    │
│   │                                │ different object, effectively "dumping" the │ different object, effectively "dumping" the │
│   │                                │ previous one.                               │ previous one.                               │
└───┴────────────────────────────────┴─────────────────────────────────────────────┴─────────────────────────────────────────────┘

### Every line
The readout at the indentation in front of each line of the function (where the figure's callouts sit), next to the
line the LM then writes.

In [7]:
def per_line_table(title, toks, starts, text, ro, lo, hi, pins=()):
    t = Table(title=title, show_lines=False)
    t.add_column(''); t.add_column('next line', max_width=44); t.add_column('GoalScope readout', max_width=80)
    for j in line_start_tokens(toks, starts, text, lo, hi):
        line = text[starts[j + 1]:].split('\n', 1)[0]
        mark = '[bold]*[/]' if j in pins else ''
        t.add_row(mark, escape(line), f'[{PAT_C}]' + escape(' '.join(ro[j].split())) + '[/]')
    console.print(t)

per_line_table('stable marriage: readout before each line (* = a figure callout)', toks_a, starts_a, full_a, ro_a,
               cs_a, fe_a + 1, pins={m[2] for m in marks})

                                 stable marriage: readout before each line (* = a figure callout)                                 
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃   ┃ next line                                    ┃ GoalScope readout                                                           ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   │  n = len(men_preferences)                    │ I'm currently processing proposals from each suitor, trying to find a       │
│   │                                              │ stable matching where each participant is matched with their most preferred │
│   │                                              │ choice, ensuring no instability exists.                                     │
│   │  men_free = list(range(n))                   │ I'm waiting for the user to provide the specific line of code or the list   │
│   │                                              │ of preferences from each participant to determine the matches.              │
│   │  women_free = list(range(n))                 │ I'm waiting for a specific action from each participant (e.g., a man or a   │
│   │                                              │ woman) to indicate their preferences or choices, typically in the context   │
│   │                                              │ of the Stable Marriage problem algorithm.                                   │
│   │  men_engaged_to = [None] * n                 │ I'm waiting for you to provide the specific line or context of the code to  │
│   │                                              │ determine who is "doing" what. Please specify which person or entity you're │
│   │                                              │ asking about.                                                               │
│   │  women_engaged_to = [None] * n               │ I'm not currently tracking any state for individuals, so I don't know what  │
│   │                                              │ line of code you're referring to. Each person is represented by their own   │
│   │                                              │ state, and I update these states as pairs form. Could you provide more      │
│   │                                              │ details about who you're asking for?                                        │
│   │  men_proposals = [0] * n                     │ I'm checking which proposals each person has made and updating their        │
│   │                                              │ preferences accordingly.                                                    │
│   │  while men_free:                             │ I'm currently processing proposals from suitors, where each suitor proposes │
│   │                                              │ to their next preferred choice until all are matched.                       │
│ * │  man = men_free.pop(0)                       │ I'm trying to match an unmatched man with a woman he hasn't proposed to     │
│   │                                              │ yet.                                                                        │
│   │  woman =                                     │ I'm evaluating the preferences of the current user and comparing them with  │
│   │ men_preferences[man][men_proposals[man]]     │ the preferences of potential matches to find the best match for them.       │
│   │  men_proposals[man] += 1                     │ I'm waiting for you to specify which line of code you're referring to.      │
│   │  if women_engaged_to[woman] is None:         │ I'm waiting for you to specify which line of code you're referring to.      │
│   │  women_engaged_to[woman] = man               │ I'm waiting for you to provide the line of code or specify which program    │
│   │                                              │ you're referring 

### Paper-style view
The viewer of the paper's qualitative figures, with the four callouts pinned in the figure's numbering. Hover or click
a token to see its readout (needs JavaScript). Also saved as a standalone page in `notebooks/demos/outputs/`.

In [8]:
u = full_a.index(FIG.USER)                   # show the user turn + code, drop the system prompt
start_tok = next(i for i, s in enumerate(starts_a) if s + len(toks_a[i]) > u)
pins = [p for num, _, p, _ in sorted(marks)]
doc = goalscope_qualitative_html([r for r in res_a if r['pos'] >= start_tok],
                                 title='GoalScope · stable marriage · 122 goal heads',
                                 save_path='notebooks/demos/outputs/01.1_stable_marriage_viewer.html', initial_pins=pins)
iframe(doc, 760)

## 2. Merge sort
The paper's merge-sort prompt, with the assistant turn prefilled with `def merge_sort` (the function name only). The LM
writes the rest, and GoalScope reads out every token of the code.

In [9]:
t0 = time.time()
MS_USER, PREFILL_MS = 'Implement the merge-sort algorithm in python', '<code>\ndef merge_sort'
prompt_b = (f"<|im_start|>system\n{FIG.SYSTEM}<|im_end|>\n<|im_start|>user\n{MS_USER}<|im_end|>\n"
            f"<|im_start|>assistant\n{PREFILL_MS}")
gen_b = greedy(prompt_b, MS_GEN)
show_code('the LM writes the function', PREFILL_MS.removeprefix('<code>\n'), gen_b)
print(f'({time.time() - t0:.0f}s)')

─────────────────────────────────────────────────── the LM writes the function ───────────────────────────────────────────────────

def merge_sort(arr):
    if len(arr) > 1:
        mid = len(arr) // 2
        L = arr[:mid]
        R = arr[mid:]

        merge_sort(L)
        merge_sort(R)

        i = j = k = 0

        while i < len(L) and j < len(R):
            if L[i] < R[j]:
                arr[k] = L[i]
                i += 1
            else:
                arr[k] = R[j]
                j += 1
            k += 1

        while i < len(L):
            arr[k] = L[i]
            i += 1
            k += 1

        while j < len(R):
            arr[k] = R[j]
            j += 1
            k += 1
</code>

(14s)


In [10]:
t0 = time.time()
full_b = prompt_b + gen_b
toks_b, starts_b, cs_b = tokens_of(full_b)
pos_b = list(range(cs_b, len(toks_b)))           # the code segment: prefill signature + every generated token
res_b = apply_goalscope(full_b, mt=mt, selected_heads=heads, include_noncode=True, token_positions=pos_b, **GS)
ro_b = {r['pos']: (r['patched_gen'] or '').strip() for r in res_b if r['scoped']}
print(f'{len(ro_b)} read-outs ({time.time() - t0:.0f}s)')

177 read-outs (226s)


In [11]:
# the figure's five callouts, transcribed from the paper:
# (callout, line, where, figure read-out); 'indent' = the indentation token before the line, 'token' = the line's first token
FIG_MS = [
    (1, 'mid = len(arr) // 2', 'indent', "I'm splitting the list into halves, recursively sorting each half, and then merging the sorted halves."),
    (2, 'merge_sort(L)', 'indent', "I'm not doing anything yet; I'm preparing to recursively sort each half of the array by dividing them into smaller subarrays, sorting those subarrays, and then merging them back together."),
    (3, 'while i < len(L) and', 'indent', "I'm comparing elements from two lists and merging them into a single sorted list."),
    (4, 'while i < len(L):', 'token', "I'm waiting for the next element from the left list to be added."),
    (5, 'while j < len(R):', 'indent', "I'm waiting for you to provide the specific line of code. Please share the remaining part of the code that needs sorting."),
]
marks_b = []
for num, line, where, ref in FIG_MS:
    idx = full_b.index(line)
    j = max(i for i, s in enumerate(starts_b) if s < idx)
    if where == 'indent':
        j -= toks_b[j].strip() != '' and full_b[starts_b[j]:idx].strip() == ''
    marks_b.append((num, line, j, ref))

t = Table(title='merge sort: GoalScope at the figure callouts', show_lines=True)
t.add_column('#'); t.add_column('token'); t.add_column('next line', max_width=24)
t.add_column('GoalScope readout', max_width=42); t.add_column('readout in the figure', max_width=42)
for num, line, p, ref in marks_b:
    t.add_row(str(num), escape(repr(toks_b[p])), escape(line), f'[{PAT_C}]{escape(ro_b[p])}[/]', escape(ref))
console.print(t)
per_line_table('merge sort: readout before each line (* = a figure callout)', toks_b, starts_b, full_b, ro_b, cs_b,
               len(toks_b), pins={m[2] for m in marks_b})

                                           merge sort: GoalScope at the figure callouts                                           
┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ token     ┃ next line            ┃ GoalScope readout                          ┃ readout in the figure                      ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ '       ' │ mid = len(arr) // 2  │ I'm splitting the list into halves,        │ I'm splitting the list into halves,        │
│   │           │                      │ recursively sorting each half, and then    │ recursively sorting each half, and then    │
│   │           │                      │ merging the sorted halves.                 │ merging the sorted halves.                 │
├───┼───────────┼──────────────────────┼────────────────────────────────────────────┼────────────────────────────────────────────┤
│ 2 │ '       ' │ merge_sort(L)        │ I'm not doing anything yet; I'm preparing  │ I'm not doing anything yet; I'm preparing  │
│   │           │                      │ to recursively sort each half of the array │ to recursively sort each half of the array │
│   │           │                      │ by dividing them into smaller subarrays,   │ by dividing them into smaller subarrays,   │
│   │           │                      │ sorting those subarrays, and then merging  │ sorting those subarrays, and then merging  │
│   │           │                      │ them back together.                        │ them back together.                        │
├───┼───────────┼──────────────────────┼────────────────────────────────────────────┼────────────────────────────────────────────┤
│ 3 │ '       ' │ while i < len(L) and │ I'm comparing elements from two lists and  │ I'm comparing elements from two lists and  │
│   │           │                      │ merging them into a single sorted list.    │ merging them into a single sorted list.    │
├───┼───────────┼──────────────────────┼────────────────────────────────────────────┼────────────────────────────────────────────┤
│ 4 │ ' while'  │ while i < len(L):    │ I'm waiting for the next element from the  │ I'm waiting for the next element from the  │
│   │           │                      │ left list to be added.                     │ left list to be added.                     │
├───┼───────────┼──────────────────────┼────────────────────────────────────────────┼────────────────────────────────────────────┤
│ 5 │ '       ' │ while j < len(R):    │ I'm waiting for you to provide the         │ I'm waiting for you to provide the         │
│   │           │                      │ specific line of code. Please share the    │ specific line of code. Please share the    │
│   │           │                      │ remaining part of the code that needs      │ remaining part of the code that needs      │
│   │           │                      │ sorting.                                   │ sorting.                                   │
└───┴───────────┴──────────────────────┴────────────────────────────────────────────┴────────────────────────────────────────────┘

                                merge sort: readout before each line (* = a figure callout)                                 
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃   ┃ next line                         ┃ GoalScope readout                                                                ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   │  if len(arr) > 1:                 │ I'm not executing any code directly; I'm dividing the task into smaller parts,   │
│   │                                   │ recursively sorting them, and then merging them back together.                   │
│ * │  mid = len(arr) // 2              │ I'm splitting the list into halves, recursively sorting each half, and then      │
│   │                                   │ merging the sorted halves.                                                       │
│   │  L = arr[:mid]                    │ I'm splitting the list into two halves and recursively sorting each half.        │
│   │  R = arr[mid:]                    │ I'm waiting for you to split the code into two halves and provide each half      │
│   │                                   │ separately so I can process them individually.                                   │
│ * │  merge_sort(L)                    │ I'm not doing anything yet; I'm preparing to recursively sort each half of the   │
│   │                                   │ array by dividing them into smaller subarrays, sorting those subarrays, and then │
│   │                                   │ merging them back together.                                                      │
│   │  merge_sort(R)                    │ I'm not doing anything; I'm waiting for you to split the code into halves, sort  │
│   │                                   │ each half, and then merge sort them. After that, I'll help merge the sorted      │
│   │                                   │ halves back together.                                                            │
│   │  i = j = k = 0                    │ I'm waiting to compare and merge sorted subarrays.                               │
│ * │  while i < len(L) and j < len(R): │ I'm comparing elements from two lists and merging them into a single sorted      │
│   │                                   │ list.                                                                            │
│   │  if L[i] < R[j]:                  │ I'm comparing elements from two lists and placing the smaller element into the   │
│   │                                   │ main list.                                                                       │
│   │  arr[k] = L[i]                    │ I'm comparing elements and merging them into the main array.                     │
│   │  i += 1                           │ I'm copying the element from the left array to the main array.                   │
│   │  else:                            │ I'm comparing elements, not executing a specific line yet.                       │
│   │  arr[k] = R[j]                    │ I'm comparing elements and placing the smaller one into the current position.    │
│   │  j += 1                           │ Executing the next instruction.                                                  │
│   │  k += 1                           │ I'm comparing elements and placing the smaller one into the main array.          │
│   │  while i < len(L):                │ I'm waiting for you to provide the specific part of the code to review and       │
│   │                                   │ merge.                                                                           │
│   │  arr[k] = L[i]                    │ I'm adding the remaining elements from the left subarray.                        │
│   │  i += 1                           │ I'm copying the current element from the left subarray to the main array.        │


### Paper-style view
The five figure callouts are pinned.

In [12]:
u = full_b.index('Implement the merge-sort')
start_tok = next(i for i, s in enumerate(starts_b) if s + len(toks_b[i]) > u)
doc = goalscope_qualitative_html([r for r in res_b if r['pos'] >= start_tok],
                                 title='GoalScope · merge sort · 122 goal heads',
                                 save_path='notebooks/demos/outputs/01.1_merge_sort_viewer.html',
                                 initial_pins=[m[2] for m in marks_b])
iframe(doc, 760)

## What to expect
<img src="figures/01.1_stable_marriage_results.png" width="400" alt="GoalScope readouts while the LM implements stable marriage (Figure 32 in the paper)">

Nothing in either prompt names the steps, yet before many lines the readout describes what the line is for: matching
a free man with a woman he has not proposed to yet, comparing the woman's preferences, splitting the list and merging
the sorted halves. Other lines, mostly setup and single increments, get generic or empty answers. The readouts at the
callouts should match the paper's figures to within a word. These are qualitative examples, not scores. See App. L.1
and Figures 31 (merge sort) and 32 (stable marriage) in the paper.